In [ ]:
import gdsfactory as gf
from axiomatic.pic_helpers import plot_circuit
import cspdk.si220.cband
from axiomatic import Axiomatic
# import ax_core.pic.pdk.pdks.cspdk_simple_models as models

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks

# Activate PDK and Axiomatic client
cspdk.si220.cband.activate_pdk()
ax_client = Axiomatic()
pdk = gf.get_active_pdk()

# Parameters
coupling_lengths = [2, 3, 4, 5, 6, 10, 15, 20]
wvl_span = 0.01 # in um
n_wvl_points = 20000
fsr = 2 # in nm
cross_section_name = "strip"
bend_radius = 5

# Use getattr to get the function from the module
# model = getattr(models, cross_section_name)
# params = model()
n_eff = 2.38
n_g = 4.3
central_wvl = 1.55
wavelengths = np.linspace(central_wvl - wvl_span/2, central_wvl + wvl_span/2, n_wvl_points)

# Target total round-trip length for FSR ≈ 1 nm
ring_length_target_fsr = central_wvl**2 / (n_g * fsr * 1e-3)  # ≈ 572.6 μm
m = round(n_eff * ring_length_target_fsr / central_wvl)
print(m)
ring_length_target = m * central_wvl / n_eff # Actual round-trip length for resonance at 1.55 um
length_arc = 2 * np.pi * bend_radius

for coupling_length in coupling_lengths:
    c = gf.Component()

    # Adjust length_y to maintain constant round-trip length
    length_x = coupling_length
    length_y = (ring_length_target - length_arc - 2 * length_x)/2

    ring = c << pdk.get_component(
        'ring_single',
        radius=bend_radius,
        gap=0.2,
        length_x=length_x,
        length_y=length_y
    )
    ring.name = "ring"

    ring.rotate(0).move((0.0, 0.0))

    # Add ports
    c.add_port("in0", port=ring.ports["o1"])
    c.add_port("out0", port=ring.ports["o2"])

    # Run Axiomatic simulation
    netlist = c.get_netlist()
    response = ax_client.pic.circuit.get_sax_spectrum(
        netlist=netlist,
        wavelengths=wavelengths.tolist() # this method does not work with arrays, but with lists only
    )
    response_db = np.array(response.spectrum_db['in0,out0'][0]) # convert it to numpy array to be able to negate it in FSR calculation
    print(f"ER: {max(response_db) - min(response_db)} dB")
    
    # Invert to find dips instead of peaks
    peaks, _ = find_peaks(-response_db, height=None, distance=10)
    # Get corresponding wavelengths
    resonance_wavelengths = wavelengths[peaks]
    # Compute FSRs
    fsrs = np.diff(resonance_wavelengths)
    avg_fsr = np.mean(fsrs)
    print(f"Average FSR: {avg_fsr*1e3} nm")

    # Monitor alignment: find closest notch to 1.55 μm
    distances = np.abs(resonance_wavelengths - central_wvl)
    nearest_idx = np.argmin(distances)
    nearest_notch_wvl = resonance_wavelengths[nearest_idx]
    wavelength_error = nearest_notch_wvl - central_wvl
    print(f"Nearest notch to 1.55 μm: {nearest_notch_wvl:.6f} μm")
    print(f"Offset from 1.55 μm: {wavelength_error * 1e3:.2f} pm")
    
    # Create individual plot
    plt.figure(figsize=(8, 5))
    plt.plot(wavelengths, response_db)
    plt.xlabel("Wavelength (μm)")
    plt.ylabel("Transmission (dB)")
    plt.title(f"Ring Resonator Spectrum\nCoupling Length = {coupling_length} μm")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

858


ApiError: status_code: 400, body: {'detail': "Netlist does not match pdk PDKType.CSPDK_SI220. Error: Invalid components {'ring_single'} for PDK PDKType.CSPDK_SI220"}